# Transformer

기존 seq2seq + attention에서는 여전히 RNN 구조이고, attention은 그 위에 붙는 형태였다

Seq2Seq에서는 인코더가 입력 문장을 다 읽고, 마지막 hidden state 같은 하나의 고정된 context vector로 전체 입력을 압축해서 디코더에 넘긴다(병목 현상의 원인)

그런데 Seq2Seq + Attention에서는 디코더가 매 시점마다 그 고정 벡터 하나만 보는 게 아니라, 인코더의 모든 hidden state를 다시 참고한다. 즉 디코더 시점 t마다 현재 필요한 입력 위치를 골라서 더 많이 참고한다. 

-> 이를 통해 병목 문제를 아느정도 해결함.

## Transformer의 등장


Attention이라는 연산만으로, Sequence Modeling이 가능하다는 아이디어
> 시퀀스 모델링에서 필요한 것은 문맥 즉 컨텍스트를 이해하는 능력이다

> x1 x2 x3.. xt 라는 시퀀스(즉, 문장의 토큰)이 있는데, 문맥을 안다 = 단어간의 관계를 알아야 한다!!

> RNN은 단어 하나하나 읽어내려가면서 필요한 정보를 ht라는 메모리에 계속 업데이트하는 방식으로 컨텍스트를 저장하게 된다. 이게 바로 RNN의 문제를 야기한 원인이 되었다.

> 근데 어텐션이라는 연산은 그 연산 자체가 global context를 반영하기 때문에 RNN 처럼 ht라는 특정 공간에 컨텍스트를 따로 저장할 필요가 없다

어텐션이라는 연산은 xt를 보고 있는데 x2와 xt에 대한 관계 정보가 필요하면 xt와 x2를 내적하면 된다. -> 어텐션 스코어 나오고, 이 값을 통해 가중치를 부여하고 가중치를 x2에 부여해서 계산하는데 사용하면? xt를 읽고 있을 때 이전에 읽고 있는 x2에 대한 정보 즉, context가 필요하면 내적을 통해 어느시점이든 이 값을 끌고 올 수 있다

언제든지 맥락(context)를 반영할 수 있다는 것이다 -> 어텐션이라는 연산 자체가 global context를 내재하고 있는 연산인다. 

**트랜스포머는 자연어 처리의 패러다임을 바꾼 구조이다**

## 트랜스포머의 RNN 한계 극복

RNN, LSTM은 시퀀스를 순차 처리 -> 1. 긴 문장에서 정보 손실(기울기 소실 -> 장기의존성) 2. 병렬처리 불가능해서 느림(GPU의 이점 누리지 못한다)

seq2 seq2는 context vector에 요약해서 디코더로 보냄 -> 아무리 긴 문장도 하나의 벡터로 압축하려다 보니 3. 정보손실

트랜스포머: RNN 없이 Attention만으로 시퀀스를 처리하여 위의 문제를 해결했다!
- 병렬 처리? -> self attention은 모든 단어 간 관계를 한 번에 계산하기에 병렬 처리 가능
- 장기의존성 -> 문장 내에서 아무리 멀리 떨어진 단어라도 직접 관계를 파악하는 것이 가능함

Transformer는 seq2seq2의 구조인 인코더 - 디코더 구조는 유지하되, 여기서 RNN을 Attention으로 완전 대체한 것임!!

## Attention 구조

### 큰 그림 - 인코더와 디코더

**인코더**: 읽는 뇌(이해를 담당)
> 입력 시퀀스를 문맥화된 표현으로 변환하여 입력 문장을 깊이 이해한다  
> 전체 입력을 동시에 처리한다(병렬)  
> 같은 구조를 6층 반복한다. 
>> 인코더는 한 층(layer)로 끝나는게 아니라 여러 layer를 여러번 통과시켜서 사용한다 트랜스포머 논문에서는 그것을 6개로 사용했다.

각 층은
1. 멀티헤드 어텐션 (여러번 셀프어텐션)
2. 피드포워드 신경망..

물론, 1층 ~ 6층은 위의 구조가 같다는 것이지, 가중치까지 똑같다는 의미는 아니다. 각 층이 배우는 파라미터는 모두 다르다. 

  - 1층에서 대략적인 관계를 보고                                                                                                                                                                                                                                
  - 위로 갈수록 더 복잡한 문맥과 의미를 반영하면서                                                                                                                                                                                                              
  - 점점 더 정교한 표현으로 바뀌어 간다,                                                                                                                                                                                                                     
                                                                                                                                                                                                                                                                
  예를 들어 "나는 어제 은행에 갔다" 같은 문장이 들어오면,                                                                                                                                                                                                       
  아래 층에서는 단어 간 기본 관계를 보고,                                                                                                                                                                                                                       
  위 층으로 갈수록 "은행"이 금융기관인지 강둑인지 같은 문맥적 의미를 더 잘 반영하게 된다.                                                                                                                                                                         
                                                                                                                                                                                                                                                                
  한 줄로 정리하면:                                                                                                                                                                                                                                             
                                                                                                                                                                                                                                                                
  “같은 구조를 6층 반복” = 같은 형태의 인코더 블록을 6개 쌓아서, 입력 표현을 단계적으로 더 깊게 이해하게 만든다. 



**디코더**: 쓰는 뇌(이해를 바탕으로 글 쓰기 담당)  
> 인코더 출력을 참조하며 시퀀스를 생성한다.  
> 한 단어씩 순서대로 출력하며 학습할 떄 미래 단어는 볼 수 없다(마스킹)

주요크기 |   임베딩차원(d_model): 512   |   어텐션헤드: 8개 |   인코더/디코더층: 각6개

> 즉, 인코더 블록을 세로로 6층 쌓음 -> 멀티헤드 어텐션이 6번 반복 / 헤드가 8개라는 것은 각 층의 멀티어텐션 내부에서 어텐션을 8개로 병럴 수행한다는 뜻이다.  

  - 어텐션(attention): 단어들이 서로 얼마나 참고할지 계산하는 방식                                                                                                                                                                                              
  - 셀프 어텐션(self-attention): 한 문장 안의 각 단어가 같은 문장 안의 다른 단어들를 참고하는 것                                                                                                                                                                
  - 멀티헤드 어텐션(multi-head attention): 이런 어텐션을 여러 세트로 나눠 동시에 수행하는 것

트랜스포머에서는 입력 문장으로부터 여러개의 head를 만들고 각 head가 각각 self attention을 수행한다. 

>RNN + attention에서는 보통 이렇게 돼:                                                                                                                                                                                                                         
                                                                                                                                                                                                                                                                
  - RNN이 입력을 순서대로 읽음                                                                                                                                                                                                                                  
  - 각 시점의 hidden state를 만듦                                                                                                                                                                                                                               
  - 디코더가 어떤 단어를 생성할 때, 인코더의 여러 hidden state 중 어디를 참고할지 attention을 계산함                                                                                                                                                            
                                                                                                                                                                                                                                                                
  즉, 전통적인 RNN + attention의 attention은 주로                                                                                                                                                                                                               
  “디코더의 현재 상태가 인코더 출력들 중 무엇을 참고할까?”                                                                                                                                                                                                      
  에 가깝다                                                                                                                                                                                                                                                    
  이건 흔히 encoder-decoder attention, 즉 cross-attention 성격이 강함                                                                                                                                                                                         
                                                                                                                                                                                                                                                                
  반면 트랜스포머의 self-attention은:                                                                                                                                                                                                                           
                                                                                                                                                                                                                                                                
  - 문장 안의 각 위치가                                                                                                                                                                                                                                         
  - 같은 문장 안의 모든 위치를 직접 참고해서                                                                                                                                                                                                                    
  - 자기 표현을 업데이트함                                                                                                                                                                                                                                      

  즉,                                                                                                                                                                                                                                                           
  i번째 단어가 j번째 단어를 얼마나 볼지를                                                                                                                                                                                                                       
  문장 내부에서 직접 계산하는 구조임                                                                                                                                                                                                                           
                                                                                                                                                                                                                                                                
  그래서 차이를 정리하면..                                                                                                                                                                                                                                 
                                                                                                                                                                                                                                                                
  1. 참고 대상이 다름                                                                                                                                                                                                                                           
                                                                                                                                                                                                                                                                
  - RNN + attention: 보통 디코더가 인코더 출력들에 주의                                                                                                                                                                                                         
  - Transformer self-attention: 각 단어가 같은 시퀀스의 다른 단어들에 주의

### 위치 인코딩(+입력 임베딩)

임베딩: 각 코튼을 512 차원의 숫자 벡터로 변환한다
> 트랜스포머 모델 안에는 임베딩도 포함되어 있다(즉, 토큰 받아서 자기가 임베딩까지 한다)  
> 토큰 ID를 받아서 임베딩 벡터로 바꾸는 층이 트랜스포머 모델의 일부로 들어있다

**위치 인코딩**(Positional Encoding): 각 위치에 고유한 신호를 '더해준다' 이를 통해 모델이 각 단어(토큰)의 순서를 알 수 있다  
RNN에서는 위치 인코딩이 필요없었다. 왜? 화살표 즉, 순환이 있으니까. 하지만, recurrent한 연결이 트랜스포머에는 없고 그 대신 위치 인코딩으로 위치 정보를 남긴다
> RNN에서는 위치 인코딩 없다. 위치 정보 입력하지 않는다.

### 어텐션 메커니즘 - Q, K, V

핵심 아이디어: 출력을 생성할 때, 입력의 어디에 주목해야하는가? -> 이 방식을 학습하자

Query - 입력벡터 x W_Q로 생성 (나는 누구와 관련 있을까????)

Key - 입력벡터 x W_k로 생성 (나는 이런 정보를 가지고 있어~ 라고 응답)

Value - 입력벡터 x W_V로 생성 (내 실제 정보는 이거야. 즉, Q와 K가 매칭되어 전달되면 곱해준다)



중요: **여기서 W_Q, W_K, W_V는 모두 학습으로 결정되는 것이다. 모델이 훈련하면서 어떤 관계를 잘 봐야 하는지를 이 세가지 가중치를 업데이트하며 스스로 학습한다**

#### self - Attention(Q,K,V) = Softmax(Q·K^T / √d_k) × V 
#### Attention Value 구하는 과정 

1. **Q와 K의 내적** -> 유사도 점수를 구한다.  
'나는'이라는 토큰의 Q를 구하고 Q와 모든 단어의 K를 내적한다: 내적의 값이 클수록 관련이 깊은 즉, 서로 연관있는 단어라는 의미이다.

2. √d_k로 나누기→스케일링 : 값이 너무 크면 소프트멕스 값이 한 곳으로 몰리기 때문에 적당한 크기로 조절해준다

3. Softmax →확률분포로 변환: 모든 점수의 합이 1이 되는 확률로 변환한다. 이때 각각 값은 어디에 더 많이 더 적게 주목해야 할자? 에 대한 가중치임

4. **가중치로 V를 곱해서 합산 (Weighted sum)** 
0.3×V("나는") + 0.7×V("학생입니다") -> Q에 해당하는 단어와 관련 정보가 반영된 **새로운 벡터가 탄생하게 되는 것이다!!**


*조금 더 자세히 살펴보자면!*  

3번에서 얻은 softmax 결과는 각 단어를 얼마나 참고할지에 대한 가중치이고, 4번 과정은 그 가중치들을 사용해서 각 단어의 V 벡터들을 가중합하는 과정이다.
  - softmax(QK^T / sqrt(d_k)) = 가중치 행렬                                                                                                                                                                                                                     
  - V = 실제로 가져올 정보                                                                                                                                                                                                                                      
  - 둘을 곱하는 것 = V들의 weighted sum

 예를 들어 "나는" 이라는 토큰을 기준으로 보면, softmax 결과가 이렇게 나왔다고 해보자:                                                                                                                                                                          
                                                                                                                                                                                                                                                                
  - "나는": 0.1
  - "어제": 0.2                                                                                                                                                                                                                                                 
  - "은행에": 0.5
  - "갔다": 0.2                                                                                                                                                                                                                                                 
                                                                                                                                                                                                                                                                
  그러면 "나는"의 새로운 표현은 대략:                                                                                                                                                                                                                           
                                                                                                                                                                                                                                                                
  0.1 * V(나는) + 0.2 * V(어제) + 0.5 * V(은행에) + 0.2 * V(갔다)


결국 Self attention을 거치 뒤에 '나는'이라는 토큰은 원래의 입력 임베딩 그 자체가 아니라 문맥이 반영된 새로운 벡터로 업데이트 된다.

즉, 토큰이 새로운 표현이 된 것이다.  **“원래 임베딩 대신, self-attention 결과로 얻은 문맥화된 벡터를 이후 층에서 사용한다”** (물론 여기서 여러 연산을 거친 결과가 다음 층으로 전달되기는 한다. )
암튼, 이러한 과정을 통해서 토큰이 contextual embedding할 수 있는 것이다.

  - "I went to the bank" 에서의 표현                                                                                                                                                                                                                            
  - "I sat by the bank of the river" 에서의 표현                                                                                                                                                                                                                
                                                                                                                                                                                                                                                                
  이 서로 달라져야 하잖아.                                                                                                                                                                                                                                      
  그래서 self-attention은 같은 단어라도 주변 문맥에 따라 다른 벡터 표현을 만들게 한다.  

### self attention에 대해

토큰끼리의 관계가 쌘가? 이게 더 정확한거 아니야? (순환보다)

RNN처럼 앞뒤연결이 아니라, 들어온 토큰들 간에 1대1 매치를 만들어서 그 매치간에 의미적 관련도를 보자는 것이다. (발상의 전환임)

이렇게 하니까 기울기 소실이 해결되었다. 
대신 더 많은 계산을 해야해서 컴퓨팅 자원이 더 많이 들지만,  앞부분에 있다고 뒤에 영향을 못미치는 것이 아니다. 앞뒤 상관 없다!

어텐션으로 쭉 돌리면 아래 그림과 같은 현상이 발생한다

it과 관계를 쌔게 맺는 것이 뭘까? -> animal 어.. 이거 사람이 이해하는거랑 똑같다! 

이런식으로 이해하는 숫자 처리라면 대단히 정확하다! -> 이게 어텐션이다.

어텐션이란? -> 우리는 문장 읽으면 어디에 어텐션을 두나? 사람이 관심을 두고 문장을 이해하는 것과 유사한 알고리즘이 된다. 그래서 이름이 어텐션 알고리즘이다. 



**일반 attention**
> Q 디코더에서 , K V 인코더에서!  
즉, 번역할 떄 원문의 어디를 볼까? 즉, 두개의 다른 시퀀스 간의 관계를 본다

**self attention**
> Q, K, V 모두 같은 시퀀스에서 온다.(그래서 self)
즉, 하나의 문장 안에서 단어 간의 관계를 본다. 한 문장 안의 관계를 모두 스스로 파악하는 것이다


self attention 이점
1. 거리 무관(왜? 아무리 멀리 떨어진 단어라도 직접 연결하니까)
2. 병렬 처리(모든 단어 쌍의 관계를 동시에 계산해서 GPU활용 가능)
3. 해석 가능성(가중치 시각화로 모델의 주목적 확인 가능)

*참고로 트랜스포머의 병럴 -> 1. 멀티헤드 8개를 동시에 수행한다!(이건 부가적인 병렬로 head 여러개를 한번에 행렬 연산으로 처리) **2. 문장 내 모든 토큰의 attention value를 한번에 처리한다(이게 가장 핵심적인 병렬, 장점이 된다)**